# 🏃 POSE LAB — MediaPipe로 배우는 포즈 추정과 관절 기하학

> **[26년 3기] NPU 활용 온디바이스 AI 프로그래밍** · 특별 세션 · **CPU 런타임으로 충분** (GPU 불필요!)

---

## 🔑 핵심 메시지

> **"무거운 검출기는 가끔, 가벼운 추적기는 매 프레임."**
>
> MediaPipe BlazePose가 휴대폰 CPU에서도 실시간인 비밀은 SAM LAB에서 본 것과 같은 **비대칭 설계**입니다:
> 사람 검출기(무겁다)는 첫 프레임과 추적 실패 시에만 돌고, 랜드마크 모델(가볍다)이 이전 프레임의 ROI를 물려받아 매 프레임을 처리합니다.
> 그리고 랜드마크가 나온 뒤부터는 **딥러닝이 아니라 기하학**입니다 — 내적 하나로 관절 각도가, 상태기계 하나로 운동 카운터가 만들어집니다.

## 📋 실습 로드맵

| Part | 주제 | 도구 | 재현성 |
|---|---|---|---|
| 1 | 33개 랜드마크 해부 | MediaPipe Tasks | 📊 |
| 2 | 관절 각도 = 내적 | **순수 numpy** | ✅ 완전 재현 |
| 3 | 스쿼트 카운터 — 임계값의 배신 | **seeded numpy** → 실영상 | ✅ → 📊 |
| 4 | z축의 정체 — 한 장 사진의 깊이 | MediaPipe | 📊 |
| 5 | 리포트 과제 | — | — |

## ⚙️ 실행 환경
- 런타임: **CPU로 충분** — 이것 자체가 오늘의 교훈입니다 (BlazePose는 모바일 CPU 타깃으로 설계된 모델)
- 전체 실행 시간: 📊 약 10~15분


---
# Part 0 · 환경 설정

In [ ]:
# [0-1] MediaPipe 설치
!pip install -q mediapipe
import mediapipe as mp
print(f"✅ mediapipe {mp.__version__}")

In [ ]:
# [0-2] 포즈 랜드마커 모델 다운로드 (Full, 약 9MB)
# 💡 수업 원칙: urllib 대신 curl -sL (리다이렉트 안전)
!curl -sL -o pose_landmarker_full.task \
  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task
!ls -lh pose_landmarker_full.task

**📊 기대 출력** — 약 `9.0M` 크기 파일. 0바이트면 재실행하세요.

> 🧑‍🏫 **강사 노트**: lite(3MB)/full(9MB)/heavy(29MB) 3종 중 full 사용. 리포트 실험 ③에서 셋을 비교합니다. 다운로드 URL은 Google 모델 저장소 — 수업 전 접속 확인 후, 실패 대비로 Drive 미러를 준비하세요.

In [ ]:
# [0-3] 실습 이미지 + 공통 임포트
import numpy as np
import cv2
import matplotlib.pyplot as plt
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

!curl -sL -o pose_sample.jpg \
  https://storage.googleapis.com/mediapipe-tasks/pose_landmarker/girl-4051811_960_720.jpg

image_bgr = cv2.imread("pose_sample.jpg")
image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
print(f"이미지 크기: {image.shape}")
plt.figure(figsize=(6,7)); plt.imshow(image); plt.axis('off'); plt.show()

> 🧑‍🏫 **강사 노트**: 공식 문서의 샘플 이미지(달리는 사람)입니다. 학생들이 **자기 전신 사진을 업로드**해서 진행하면 몰입도가 크게 오릅니다 — `from google.colab import files; files.upload()` 셀을 안내하세요. 어떤 이미지든 이후 셀은 동일 동작합니다 (단, 전신이 나와야 다리 관절 실습이 가능).

---
# Part 1 · 33개 랜드마크 해부

BlazePose는 COCO의 17개가 아니라 **33개** 랜드마크를 씁니다. 얼굴 11개 + 몸통·팔 12개 + 손끝 4개 + 다리·발 10개(발뒤꿈치/발끝 포함) — 피트니스 응용을 위해 발 방향까지 추정하도록 확장된 토폴로지입니다.

각 랜드마크는 4개의 값을 가집니다:

| 필드 | 의미 | 범위 |
|---|---|---|
| `x, y` | 이미지 내 정규화 좌표 | 0~1 |
| `z` | **엉덩이 중점 기준 상대 깊이** (카메라 쪽이 음수) | 대략 -1~1 |
| `visibility` | 이 점이 화면에서 보일 확률 | 0~1 |

> 단안 카메라인데 z가 나온다? — Part 4에서 이 마법의 정체와 한계를 실험합니다.

In [ ]:
# [1-1] 랜드마커 로드 + 이미지 1장 추론
base = mp_python.BaseOptions(model_asset_path="pose_landmarker_full.task")
opts = vision.PoseLandmarkerOptions(base_options=base, output_segmentation_masks=False)
landmarker = vision.PoseLandmarker.create_from_options(opts)

mp_img = mp.Image.create_from_file("pose_sample.jpg")

import time
t0 = time.perf_counter()
result = landmarker.detect(mp_img)
t_ms = (time.perf_counter()-t0)*1000

lms = result.pose_landmarks[0]        # 첫 번째(유일한) 사람
print(f"검출된 사람 수 : {len(result.pose_landmarks)}")
print(f"랜드마크 수    : {len(lms)}")            # ✅ 33
print(f"추론 시간(CPU) : {t_ms:.0f} ms  📊 (Colab CPU 기준 대략 40~150ms, 첫 회는 초기화 포함으로 더 김)")

In [ ]:
# [1-2] 랜드마크 테이블 — 주요 관절만 추려 보기
NAMES = {0:"코", 11:"왼어깨", 12:"오른어깨", 13:"왼팔꿈치", 14:"오른팔꿈치",
         15:"왼손목", 16:"오른손목", 23:"왼엉덩이", 24:"오른엉덩이",
         25:"왼무릎", 26:"오른무릎", 27:"왼발목", 28:"오른발목"}
print(f"{'idx':>3} {'이름':<8} {'x':>7} {'y':>7} {'z':>7} {'vis':>6}")
for i, name in NAMES.items():
    p = lms[i]
    print(f"{i:>3} {name:<8} {p.x:7.3f} {p.y:7.3f} {p.z:7.3f} {p.visibility:6.3f}")
print()
print("📊 관찰: visibility가 낮은 관절이 있나요? 그 관절은 실제로 가려져 있나요?")

In [ ]:
# [1-3] 스켈레톤 직접 그리기 — 연결 규칙은 '지식'이 아니라 '데이터'
# 뼈대 연결 목록 (BlazePose 토폴로지의 몸통/팔/다리 부분)
BONES = [(11,12),(11,13),(13,15),(12,14),(14,16),          # 어깨-팔
         (11,23),(12,24),(23,24),                           # 몸통
         (23,25),(25,27),(24,26),(26,28),                   # 다리
         (27,29),(29,31),(28,30),(30,32)]                   # 발

h, w = image.shape[:2]
canvas = image.copy()
for a, b in BONES:
    pa = (int(lms[a].x*w), int(lms[a].y*h))
    pb = (int(lms[b].x*w), int(lms[b].y*h))
    cv2.line(canvas, pa, pb, (57,217,138), 3)
for i in range(33):
    p = (int(lms[i].x*w), int(lms[i].y*h))
    # visibility를 점 크기에 반영 — 낮으면 작게
    r = max(2, int(6*lms[i].visibility))
    cv2.circle(canvas, p, r, (77,201,255), -1)

plt.figure(figsize=(6,7)); plt.imshow(canvas); plt.axis('off')
plt.title("33 landmarks + 16 bones (점 크기 = visibility)"); plt.show()

---
# Part 2 · 관절 각도 = 내적 — 여기서부터는 순수 기하학

포즈 추정의 진짜 힘은 랜드마크 **이후**에 나옵니다. 세 점 (a, b, c)에서 꼭짓점 b의 각도는:

$$\theta = \arccos\left(\frac{\vec{ba} \cdot \vec{bc}}{|\vec{ba}||\vec{bc}|}\right)$$

딥러닝이 아니라 **고등학교 벡터**입니다. 아래는 seeded도 필요 없는 완전 결정적 계산 — 모든 값이 ✅ 고정입니다.

In [ ]:
# [2-1] 관절 각도 함수 — 이 랩의 심장 (순수 numpy)
def joint_angle(a, b, c):
    """b를 꼭짓점으로 하는 ∠abc (도 단위)"""
    a, b, c = map(np.asarray, (a, b, c))
    v1, v2 = a - b, c - b
    cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return float(np.degrees(np.arccos(np.clip(cos, -1, 1))))

# 손계산 검증 3종 — hip → knee → ankle 좌표 (정규화 좌표계)
A = joint_angle([0.50,0.30], [0.50,0.55], [0.50,0.80])   # 일직선
B = joint_angle([0.50,0.30], [0.50,0.55], [0.75,0.55])   # 수직+수평
C_ = joint_angle([0.58,0.36], [0.50,0.55], [0.55,0.80])  # 실전형 굽힘
print(f"A 일직선 다리     : {A:.4f}°")
print(f"B 직각            : {B:.4f}°")
print(f"C 실전형 무릎 굽힘: {C_:.4f}°")

**✅ 기대 출력** (완전 결정적 — 한 자리도 다르면 안 됩니다)
```
A 일직선 다리     : 180.0000°
B 직각            : 90.0000°
C 실전형 무릎 굽힘: 145.8564°
```

**손계산 체크 (B)**: $\vec{ba}=(0,-0.25)$, $\vec{bc}=(0.25,0)$ → 내적 $=0$ → $\cos\theta=0$ → $\theta=90°$. 종이에 그려서 확인하세요.

> ⚠️ `np.clip(cos, -1, 1)`을 빼면? 부동소수점 오차로 $\cos\theta = 1.0000000002$ 같은 값이 나와 `arccos`가 **NaN**을 뱉는 날이 옵니다. Day 5에서 배운 "간헐적 NaN" 유형의 미니 버전 — 리포트 실험 ①에서 재현합니다.

In [ ]:
# [2-2] 실제 사진에 적용 — 무릎 각도 측정
def lm_xy(lms, i):
    return [lms[i].x, lms[i].y]

left_knee  = joint_angle(lm_xy(lms,23), lm_xy(lms,25), lm_xy(lms,27))
right_knee = joint_angle(lm_xy(lms,24), lm_xy(lms,26), lm_xy(lms,28))
left_elbow = joint_angle(lm_xy(lms,11), lm_xy(lms,13), lm_xy(lms,15))
print(f"왼무릎  : {left_knee:6.1f}°   📊 (달리는 자세라면 굽힘이 보여야 함)")
print(f"오른무릎: {right_knee:6.1f}°")
print(f"왼팔꿈치: {left_elbow:6.1f}°")

canvas2 = canvas.copy()
kx, ky = int(lms[25].x*w), int(lms[25].y*h)
cv2.putText(canvas2, f"{left_knee:.0f} deg", (kx+10, ky),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,180,84), 2)
plt.figure(figsize=(6,7)); plt.imshow(canvas2); plt.axis('off'); plt.show()

---
# Part 3 · 스쿼트 카운터 — 임계값의 배신 🎬

무릎 각도를 시간축으로 보면 스쿼트는 파형입니다: 서면 ~135°, 앉으면 ~45°.
"90° 아래로 내려가면 1회"라는 **단일 임계값** 카운터 — 직관적이죠. 그런데 랜드마크에는 **지터(노이즈)**가 있습니다.

먼저 **seeded 합성 신호**로 카운터 로직만 분리해 검증합니다 (✅ 완전 재현) — 우리 과정의 "로직 검증은 결정적 환경에서 먼저" 원칙 그대로입니다.

In [ ]:
# [3-1] 합성 스쿼트 신호 — 정답을 아는 실험실
rng = np.random.default_rng(42)                    # seeded!
t = np.linspace(0, 5*2*np.pi, 300)                 # 정확히 5회 스쿼트
clean = 90 + 45*np.cos(t)                          # 45°~135° 왕복
sig = clean + rng.normal(0, 4.0, 300)              # 랜드마크 지터 ±4°

plt.figure(figsize=(13,3.5))
plt.plot(sig, lw=1, color='#4dc9ff', label='측정 각도(노이즈 포함)')
plt.plot(clean, lw=1, ls='--', color='gray', alpha=.5, label='실제 각도')
plt.axhline(90, color='#ff5d5d', lw=1, label='나이브 임계값 90°')
plt.axhline(75, color='#39d98a', lw=1, ls=':', label='히스테리시스 75°/105°')
plt.axhline(105, color='#39d98a', lw=1, ls=':')
plt.legend(loc='upper right', fontsize=8); plt.ylabel('무릎 각도(°)')
plt.title('스쿼트 5회 — 그런데 카운터는 몇을 셀까?'); plt.show()

In [ ]:
# [3-2] 대결: 단일 임계값 vs 히스테리시스 상태기계
# ① 나이브 — 90° 하향 교차를 셈
naive = int(np.sum((sig[:-1] >= 90) & (sig[1:] < 90)))

# ② 히스테리시스 — 75° 미만이어야 DOWN, 105° 초과해야 UP(이때 +1)
state, hyst = "UP", 0
for v in sig:
    if state == "UP" and v < 75:
        state = "DOWN"
    elif state == "DOWN" and v > 105:
        state = "UP"; hyst += 1

print("┌────────────────────────────────────┐")
print(f"│ 실제 스쿼트 횟수      :  5         │")
print(f"│ ① 나이브(90° 교차)    :  {naive}         │")
print(f"│ ② 히스테리시스        :  {hyst}         │")
print("└────────────────────────────────────┘")

**✅ 기대 출력** (seed=42 고정 — 정확히 이 값이어야 합니다)
```
│ 실제 스쿼트 횟수      :  5         │
│ ① 나이브(90° 교차)    :  6         │
│ ② 히스테리시스        :  5         │
```

### 🎬 클라이맥스 — 왜 나이브는 6을 셌나

신호가 90° 근처를 지날 때 노이즈가 경계를 **위아래로 두 번** 넘나든 지점이 있습니다 (그래프에서 90° 선 근처를 확대해 찾아보세요). 단 ±4°의 지터가 카운트를 오염시켰습니다 — 운동 앱이 스쿼트 5개를 6개로 세는 순간, 사용자의 신뢰는 끝납니다.

히스테리시스는 **"내려갔다"와 "올라왔다"의 기준을 분리**(75°/105°)해서, 경계 근처의 떨림이 상태를 못 바꾸게 만듭니다. 슈미트 트리거(전자회로), 온도조절기 — 임베디드 엔지니어링의 고전 패턴이 그대로 비전에 적용된 것입니다.

In [ ]:
# [3-3] 실영상 적용 — 직접 찍은 스쿼트 영상으로
# 휴대폰으로 측면에서 스쿼트 3~5회를 찍어 업로드하세요 (5~10초, 전신)
from google.colab import files
up = files.upload()                      # 📊 업로드 파일명은 각자 다름
video_path = list(up.keys())[0]

base = mp_python.BaseOptions(model_asset_path="pose_landmarker_full.task")
opts = vision.PoseLandmarkerOptions(base_options=base,
                                    running_mode=vision.RunningMode.VIDEO)
vlandmarker = vision.PoseLandmarker.create_from_options(opts)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
angles, ts = [], 0
while True:
    ok, frame = cap.read()
    if not ok: break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mpf = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    res = vlandmarker.detect_for_video(mpf, int(ts))
    if res.pose_landmarks:
        L = res.pose_landmarks[0]
        angles.append(joint_angle([L[23].x,L[23].y],[L[25].x,L[25].y],[L[27].x,L[27].y]))
    ts += 1000.0/fps
cap.release()
angles = np.array(angles)
print(f"프레임 수: {len(angles)}  📊")

# 같은 두 카운터를 실측 신호에 적용
naive_r = int(np.sum((angles[:-1] >= 90) & (angles[1:] < 90)))
state, hyst_r = "UP", 0
for v in angles:
    if state=="UP" and v<75: state="DOWN"
    elif state=="DOWN" and v>105: state="UP"; hyst_r += 1
print(f"나이브: {naive_r}회 · 히스테리시스: {hyst_r}회  📊 실제 횟수와 비교해 보세요")

plt.figure(figsize=(13,3.5)); plt.plot(angles, color='#4dc9ff')
plt.axhline(75, color='#39d98a', ls=':'); plt.axhline(105, color='#39d98a', ls=':')
plt.title(f"내 스쿼트 파형 — 히스테리시스 카운트 {hyst_r}회"); plt.ylabel("무릎 각도(°)"); plt.show()

> 🧑‍🏫 **강사 노트 (양방향)**: 실영상에서는 나이브와 히스테리시스가 **같은 값이 나올 수도 있습니다** (지터가 작거나 동작이 크면). 같으면 — "합성 실험이 왜 필요했는지"의 근거로 활용하세요: 실패 조건을 통제해서 만들 수 있는 게 시뮬레이션의 가치입니다. 다르면 — 그 자리에서 90° 근처 파형을 확대해 이중 교차 지점을 찾는 라이브 디버깅이 최고의 순간이 됩니다. 촬영각이 정측면이 아니면 각도 절대값이 달라진다는 점(2D 투영의 한계 → Part 4 연결)도 짚어주세요.

---
# Part 4 · z축의 정체 — 한 장 사진에서 깊이가 나온다고?

`z`는 단안 이미지에서 **학습으로 추정된 상대 깊이**입니다 (엉덩이 중점 기준, 카메라 쪽 음수). 스테레오도 LiDAR도 없이 — 사람 신체 비례라는 강력한 사전지식(prior)을 모델이 학습한 결과입니다.

`pose_world_landmarks`는 한 걸음 더 나가 **미터 단위 3D 좌표**(엉덩이 원점)를 줍니다.

In [ ]:
# [4-1] image z vs world 좌표 비교
wl = result.pose_world_landmarks[0]
print(f"{'관절':<8} {'image z':>9} {'world x':>9} {'world y':>9} {'world z':>9}")
for i, name in [(15,"왼손목"),(16,"오른손목"),(25,"왼무릎"),(27,"왼발목")]:
    print(f"{name:<8} {lms[i].z:9.3f} {wl[i].x:9.3f} {wl[i].y:9.3f} {wl[i].z:9.3f}")

# 신체 치수 검증 — world 좌표가 정말 미터라면?
import numpy.linalg as LA
thigh = LA.norm(np.array([wl[23].x,wl[23].y,wl[23].z]) -
                np.array([wl[25].x,wl[25].y,wl[25].z]))
shin  = LA.norm(np.array([wl[25].x,wl[25].y,wl[25].z]) -
                np.array([wl[27].x,wl[27].y,wl[27].z]))
print(f"\n허벅지 길이 추정: {thigh*100:.1f} cm  📊 (성인 기준 대략 35~50cm면 그럴듯)")
print(f"정강이 길이 추정: {shin*100:.1f} cm  📊")
print("\n📊 토론: 이 값을 '측정'이라 불러도 될까요? — 학습된 신체 비례의 '추론'입니다.")
print("   카메라와의 절대 거리는 여전히 알 수 없습니다 (스케일 모호성).")

In [ ]:
# [4-2] 3D 스켈레톤 인터랙티브 회전
import plotly.graph_objects as go
xs=[wl[i].x for i in range(33)]; ys=[wl[i].y for i in range(33)]; zs=[wl[i].z for i in range(33)]
BONES = [(11,12),(11,13),(13,15),(12,14),(14,16),(11,23),(12,24),(23,24),
         (23,25),(25,27),(24,26),(26,28)]
lines=[]
for a,b in BONES:
    lines.append(go.Scatter3d(x=[xs[a],xs[b]], y=[zs[a],zs[b]], z=[-ys[a],-ys[b]],
                 mode='lines', line=dict(color='#39d98a', width=5), showlegend=False))
pts = go.Scatter3d(x=xs, y=zs, z=[-v for v in ys], mode='markers',
                   marker=dict(size=4, color='#4dc9ff'), showlegend=False)
go.Figure(data=lines+[pts]).update_layout(
    title="드래그해서 회전 — 단안 사진 한 장에서 나온 3D",
    scene=dict(aspectmode='data'), height=520).show()

---
# Part 5 · 리포트 과제 (3종 중 2종 선택)

### 실험 ① — clip 없는 arccos의 함정 재현
`joint_angle`에서 `np.clip`을 제거하고, **세 점이 정확히 일직선**인 입력을 다양한 좌표 스케일로 넣어보세요 (예: `[1e5,3e5]` 단위 픽셀 좌표). NaN이 나오는 입력을 하나 이상 찾아 원인을 부동소수점 관점에서 설명하세요.

### 실험 ② — 히스테리시스 폭의 트레이드오프
`[3-1]` 합성 신호에서 히스테리시스 폭을 (90/90), (85/95), (75/105), (60/120)로 바꿔가며 카운트를 기록하세요.
- 폭이 0이면 나이브와 같아지고, 너무 넓으면 무엇이 망가지나요? (힌트: 얕은 스쿼트) — "노이즈 내성 ↔ 감도"의 트레이드오프를 표로 정리
- 노이즈 σ를 4→8→12로 키우면 각 설정이 언제 무너지는지도 함께

### 실험 ③ — lite / full / heavy 3종 비교
모델 3종을 같은 이미지 10장에 돌려 (a) 평균 추론 시간 (b) 주요 관절 좌표의 표준편차를 비교하세요. "heavy가 항상 정답"인가요? 우리 과정의 **Edge Trilemma**(정확도·속도·전력)를 이 결과 위에서 논하세요.

---

## ✅ 체크포인트 — 오늘 확보해야 할 3가지
1. **각도 3종 고정값** `[2-1]` — 180.0000 / 90.0000 / 145.8564
2. **카운터 대결 결과** `[3-2]` — 나이브 6 vs 히스테리시스 5 (실제 5)
3. **내 스쿼트 파형** `[3-3]` — 실영상 각도 그래프 + 카운트

> 🔗 **NPU 파이프라인과의 연결 (교육자 노트)**
> BlazePose는 처음부터 **모바일 CPU에서 실시간**을 목표로 설계된 모델입니다 — 검출기/추적기 분리, 경량 백본, 고정 입력 크기. 즉 우리가 Day 2~3에서 배운 온디바이스 설계 원칙들의 **교과서적 구현체**입니다. 그리고 오늘의 후반부(각도·상태기계)는 전부 랜드마크 이후의 **후처리**였죠 — YOLO의 NMS처럼, 포즈의 각도 계산도 NPU 밖 CPU에서 도는 로직입니다. "모델이 끝나는 곳에서 제품이 시작된다"는 감각, 그것이 이 랩의 마지막 메시지입니다.